# Chinook Music Store Exploration  
**Dataset:** [Chinook Dataset (Github)](https://github.com/lerocha/chinook-database)   
*Exploratory Data Analysis*

In [ ]:
## re
import os
from dotenv import load_dotenv

load_dotenv()
%load_ext sql
%sql $DATABASE_URL

%config SqlMagic.feedback = False
%config SqlMagic.displaylimit = None

In [3]:
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [4]:
%%sql
SELECT t.table_name, 
     (SELECT COUNT(*)
     FROM information_schema.columns c
     WHERE c.table_name = t.table_name) AS column_count,
     pg_size_pretty(pg_total_relation_size(quote_ident(t.table_name))) AS total_size
FROM information_schema.tables t
WHERE t.table_schema = 'public'
ORDER BY t.table_name
LIMIT 10;

table_name,column_count,total_size
album,3,80 kB
artist,2,56 kB
customer,13,72 kB
employee,15,40 kB
genre,2,24 kB
invoice,9,120 kB
invoice_line,5,336 kB
media_type,2,24 kB
playlist,2,24 kB
playlist_track,2,936 kB


**COLUMN DATA TYPES AND SCHEMA CONFIRMATION**

In [13]:
%%sql
SELECT column_name, data_type, character_maximum_length
FROM information_schema.columns
WHERE table_name = 'album';

column_name,data_type,character_maximum_length
album_id,integer,None
artist_id,integer,None
title,character varying,160


In [14]:
%%sql
SELECT column_name, data_type, character_maximum_length
FROM information_schema.columns
WHERE table_name = 'artist';

column_name,data_type,character_maximum_length
artist_id,integer,None
name,character varying,120


In [6]:
%%sql
SELECT table_name,
       column_name, 
       data_type, 
       is_nullable,
       COUNT(column_name) OVER(PARTITION BY table_name) AS total_columns
FROM information_schema.columns
WHERE table_schema = 'public'
ORDER BY  table_name, column_name
LIMIT 10;

table_name,column_name,data_type,is_nullable,total_columns
album,album_id,integer,NO,3
album,artist_id,integer,NO,3
album,title,character varying,NO,3
artist,artist_id,integer,NO,2
artist,name,character varying,YES,2
customer,address,character varying,YES,13
customer,city,character varying,YES,13
customer,company,character varying,YES,13
customer,country,character varying,YES,13
customer,customer_id,integer,NO,13


**all tables in the database**

In [7]:
%%sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;

table_name
album
artist
customer
employee
genre
invoice
invoice_line
media_type
playlist
playlist_track


In [8]:
%%sql
SELECT name, COUNT(name) AS duplicate_count
FROM genre
GROUP BY name
HAVING COUNT(name)>1 
;

name,duplicate_count


**CUSTOMERS WHERE THE COMPANY COLUMN IS MISSING**

In [9]:
%%sql
SELECT first_name, last_name, company
FROM customer
WHERE company IS NULL
LIMIT 10;

first_name,last_name,company
Leonie,Köhler,None
François,Tremblay,None
Bjørn,Hansen,None
Helena,Holý,None
Astrid,Gruber,None
Daan,Peeters,None
Kara,Nielsen,None
Fernanda,Ramos,None
Michelle,Brooks,None
Dan,Miller,None


**CHECKING FOR NULL VALUES**

In [71]:
%%sql
SELECT 
    --# Numeric / ID Keys (Standard NULL check)
    COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_ids,
    COUNT(*) FILTER (WHERE support_rep_id IS NULL) AS missing_rep,

    -- #Text Columns
    COUNT(*) FILTER (
        WHERE first_name IS NULL 
           OR TRIM(first_name) IN ('', 'NA', 'N/A', 'Unknown', 'None') ) AS missing_customer_names,
    COUNT(*) FILTER (
        WHERE company IS NULL 
           OR TRIM(company) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_company_names,
    COUNT(*) FILTER (
        WHERE address IS NULL 
           OR TRIM(address) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_address,
    COUNT(*) FILTER (
        WHERE city IS NULL 
           OR TRIM(city) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_city,
    COUNT(*) FILTER (
        WHERE state IS NULL 
           OR TRIM(state) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_state,
    COUNT(*) FILTER (
        WHERE country IS NULL 
           OR TRIM(country) IN ('', 'NA', 'N/A', 'Unknown', 'None') ) AS missing_country,
    COUNT(*) FILTER (
        WHERE postal_code IS NULL 
           OR TRIM(postal_code) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_postal_code
FROM customer;

missing_ids,missing_rep,missing_customer_names,missing_company_names,missing_address,missing_city,missing_state,missing_country,missing_postal_code
0,0,0,49,0,0,29,0,4


In [72]:
%%sql
SELECT 'SELECT ' || 
    string_agg(
        CASE 
-- Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'album' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE album_id IS NULL) AS missing_album_id, COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_title, COUNT(*) FILTER (WHERE artist_id IS NULL) AS missing_artist_id FROM album;"


In [74]:
%%sql
--#generated_profiling_query
SELECT COUNT(*) FILTER (WHERE album_id IS NULL) AS missing_album_id, 
COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_title, 
COUNT(*) FILTER (WHERE artist_id IS NULL) AS missing_artist_id 
FROM album;


missing_album_id,missing_title,missing_artist_id
0,0,0


In [75]:
%%sql
SELECT 'SELECT ' || 
    string_agg(
        CASE 
            -- Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
            -- Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'artist' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE artist_id IS NULL) AS missing_artist_id, COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name FROM artist;"


In [76]:
%%sql
--#generated_profiling_query
SELECT COUNT(*) FILTER (WHERE artist_id IS NULL) AS missing_artist_id, 
COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name 
FROM artist;


missing_artist_id,missing_name
0,0


In [77]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'track' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id, COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name, COUNT(*) FILTER (WHERE album_id IS NULL) AS missing_album_id, COUNT(*) FILTER (WHERE media_type_id IS NULL) AS missing_media_type_id, COUNT(*) FILTER (WHERE genre_id IS NULL) AS missing_genre_id, COUNT(*) FILTER (WHERE composer IS NULL OR TRIM(composer::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_composer, COUNT(*) FILTER (WHERE milliseconds IS NULL) AS missing_milliseconds, COUNT(*) FILTER (WHERE bytes IS NULL) AS missing_bytes, COUNT(*) FILTER (WHERE unit_price IS NULL) AS missing_unit_price, COUNT(*) FILTER (WHERE minutes IS NULL) AS missing_minutes FROM track;"


In [78]:
%%sql
-- generated_profiling_query

SELECT COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id, 
COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name, 
COUNT(*) FILTER (WHERE album_id IS NULL) AS missing_album_id, 
COUNT(*) FILTER (WHERE media_type_id IS NULL) AS missing_media_type_id, 
COUNT(*) FILTER (WHERE genre_id IS NULL) AS missing_genre_id, 
COUNT(*) FILTER (WHERE composer IS NULL OR TRIM(composer::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS  missing_composer, 
COUNT(*) FILTER (WHERE milliseconds IS NULL) AS missing_milliseconds, 
COUNT(*) FILTER (WHERE bytes IS NULL) AS missing_bytes, 
COUNT(*) FILTER (WHERE unit_price IS NULL) AS missing_unit_price, 
COUNT(*) FILTER (WHERE minutes IS NULL) AS missing_minutes FROM track;


missing_track_id,missing_name,missing_album_id,missing_media_type_id,missing_genre_id,missing_composer,missing_milliseconds,missing_bytes,missing_unit_price,missing_minutes
0,0,0,0,0,977,0,0,0,0


Note: composer missing 977 of 3,503 tracks (~28% of the catalog) is a large enough gap that it will be left as NULL rather than filled with a placeholder, but it should be noted wherever a future query groups, filters, or aggregates by composer

In [79]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'employee' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE employee_id IS NULL) AS missing_employee_id, COUNT(*) FILTER (WHERE last_name IS NULL OR TRIM(last_name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_last_name, COUNT(*) FILTER (WHERE first_name IS NULL OR TRIM(first_name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_first_name, COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_title, COUNT(*) FILTER (WHERE reports_to IS NULL) AS missing_reports_to, COUNT(*) FILTER (WHERE birth_date IS NULL) AS missing_birth_date, COUNT(*) FILTER (WHERE hire_date IS NULL) AS missing_hire_date, COUNT(*) FILTER (WHERE address IS NULL OR TRIM(address::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_address, COUNT(*) FILTER (WHERE city IS NULL OR TRIM(city::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_city, COUNT(*) FILTER (WHERE state IS NULL OR TRIM(state::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_state, COUNT(*) FILTER (WHERE country IS NULL OR TRIM(country::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_country, COUNT(*) FILTER (WHERE postal_code IS NULL OR TRIM(postal_code::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_postal_code, COUNT(*) FILTER (WHERE phone IS NULL OR TRIM(phone::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_phone, COUNT(*) FILTER (WHERE fax IS NULL OR TRIM(fax::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_fax, COUNT(*) FILTER (WHERE email IS NULL OR TRIM(email::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_email FROM employee;"


In [80]:
%%sql
-- generated_profiling_query
SELECT COUNT(*) FILTER (WHERE employee_id IS NULL) AS missing_employee_id, 
COUNT(*) FILTER (WHERE last_name IS NULL OR TRIM(last_name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_last_name, 
COUNT(*) FILTER (WHERE first_name IS NULL OR TRIM(first_name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_first_name, 
COUNT(*) FILTER (WHERE title IS NULL OR TRIM(title::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_title, 
COUNT(*) FILTER (WHERE reports_to IS NULL) AS missing_reports_to, 
COUNT(*) FILTER (WHERE birth_date IS NULL) AS missing_birth_date, 
COUNT(*) FILTER (WHERE hire_date IS NULL) AS missing_hire_date, 
COUNT(*) FILTER (WHERE address IS NULL OR TRIM(address::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_address, 
COUNT(*) FILTER (WHERE city IS NULL OR TRIM(city::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_city, 
COUNT(*) FILTER (WHERE state IS NULL OR TRIM(state::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_state, 
COUNT(*) FILTER (WHERE country IS NULL OR TRIM(country::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_country, 
COUNT(*) FILTER (WHERE postal_code IS NULL OR TRIM(postal_code::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_postal_code, 
COUNT(*) FILTER (WHERE phone IS NULL OR TRIM(phone::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_phone, 
COUNT(*) FILTER (WHERE fax IS NULL OR TRIM(fax::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_fax, COUNT(*) FILTER (WHERE email IS NULL OR TRIM(email::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_email 
FROM employee;


missing_employee_id,missing_last_name,missing_first_name,missing_title,missing_reports_to,missing_birth_date,missing_hire_date,missing_address,missing_city,missing_state,missing_country,missing_postal_code,missing_phone,missing_fax,missing_email
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [81]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'genre' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE genre_id IS NULL) AS missing_genre_id, COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name FROM genre;"


In [83]:
%%sql
-- generated_profiling_query
SELECT COUNT(*) FILTER (WHERE genre_id IS NULL) AS missing_genre_id, 
COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name 
FROM genre;


missing_genre_id,missing_name
0,0


In [84]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'invoice' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE invoice_id IS NULL) AS missing_invoice_id, COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_id, COUNT(*) FILTER (WHERE invoice_date IS NULL) AS missing_invoice_date, COUNT(*) FILTER (WHERE billing_address IS NULL OR TRIM(billing_address::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_address, COUNT(*) FILTER (WHERE billing_city IS NULL OR TRIM(billing_city::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_city, COUNT(*) FILTER (WHERE billing_state IS NULL OR TRIM(billing_state::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_state, COUNT(*) FILTER (WHERE billing_country IS NULL OR TRIM(billing_country::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_country, COUNT(*) FILTER (WHERE billing_postal_code IS NULL OR TRIM(billing_postal_code::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_postal_code, COUNT(*) FILTER (WHERE total IS NULL) AS missing_total FROM invoice;"


In [85]:
%%sql
-- #generated_profiling_query
SELECT COUNT(*) FILTER (WHERE invoice_id IS NULL) AS missing_invoice_id, 
COUNT(*) FILTER (WHERE customer_id IS NULL) AS missing_customer_id, 
COUNT(*) FILTER (WHERE invoice_date IS NULL) AS missing_invoice_date, 
COUNT(*) FILTER (WHERE billing_address IS NULL OR TRIM(billing_address::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_address, 
COUNT(*) FILTER (WHERE billing_city IS NULL OR TRIM(billing_city::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_city, 
COUNT(*) FILTER (WHERE billing_state IS NULL OR TRIM(billing_state::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_state, 
COUNT(*) FILTER (WHERE billing_country IS NULL OR TRIM(billing_country::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_country, 
COUNT(*) FILTER (WHERE billing_postal_code IS NULL OR TRIM(billing_postal_code::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_billing_postal_code, 
COUNT(*) FILTER (WHERE total IS NULL) AS missing_total 
FROM invoice;


missing_invoice_id,missing_customer_id,missing_invoice_date,missing_billing_address,missing_billing_city,missing_billing_state,missing_billing_country,missing_billing_postal_code,missing_total
0,0,0,0,0,202,0,28,0


In [86]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'invoice_line' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE invoice_line_id IS NULL) AS missing_invoice_line_id, COUNT(*) FILTER (WHERE invoice_id IS NULL) AS missing_invoice_id, COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id, COUNT(*) FILTER (WHERE unit_price IS NULL) AS missing_unit_price, COUNT(*) FILTER (WHERE quantity IS NULL) AS missing_quantity FROM invoice_line;"


In [87]:
%%sql
-- #generated_profiling_query
SELECT COUNT(*) FILTER (WHERE invoice_line_id IS NULL) AS missing_invoice_line_id, COUNT(*) FILTER (WHERE invoice_id IS NULL) AS missing_invoice_id, 
COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id, 
COUNT(*) FILTER (WHERE unit_price IS NULL) AS missing_unit_price, 
COUNT(*) FILTER (WHERE quantity IS NULL) AS missing_quantity 
FROM invoice_line;


missing_invoice_line_id,missing_invoice_id,missing_track_id,missing_unit_price,missing_quantity
0,0,0,0,0


In [88]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'media_type' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE media_type_id IS NULL) AS missing_media_type_id, COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name FROM media_type;"


In [89]:
%%sql
-- #generated_profiling_query
SELECT COUNT(*) FILTER (WHERE media_type_id IS NULL) AS missing_media_type_id, 
COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name 
FROM media_type;


missing_media_type_id,missing_name
0,0


In [90]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'playlist' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE playlist_id IS NULL) AS missing_playlist_id, COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name FROM playlist;"


In [91]:
%%sql
-- #generated_profiling_query
SELECT COUNT(*) FILTER (WHERE playlist_id IS NULL) AS missing_playlist_id, 
COUNT(*) FILTER (WHERE name IS NULL OR TRIM(name::text) IN ('', 'NA', 'N/A', 'Unknown', 'None')) AS missing_name 
FROM playlist;


missing_playlist_id,missing_name
0,0


In [92]:
%%sql
SELECT 'SELECT ' || string_agg(
        CASE 
-- #Text columns: Check NULLs + Sentinels + Empty Strings
            WHEN data_type IN ('character varying', 'text', 'character') THEN
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL OR TRIM(' || column_name || '::text) IN ('''', ''NA'', ''N/A'', ''Unknown'', ''None'')) AS missing_' || column_name
-- #Non-text columns: Simple NULL check
            ELSE 
                'COUNT(*) FILTER (WHERE ' || column_name || ' IS NULL) AS missing_' || column_name
        END, 
        ', '
    ) || ' FROM ' || table_name || ';' AS generated_profiling_query
FROM information_schema.columns
WHERE table_schema = 'public' 
  AND table_name = 'playlist_track' -- Change table_name here
GROUP BY table_name;

generated_profiling_query
"SELECT COUNT(*) FILTER (WHERE playlist_id IS NULL) AS missing_playlist_id, COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id FROM playlist_track;"


In [93]:
%%sql
-- #generated_profiling_query
SELECT COUNT(*) FILTER (WHERE playlist_id IS NULL) AS missing_playlist_id, 
COUNT(*) FILTER (WHERE track_id IS NULL) AS missing_track_id 
FROM playlist_track;


missing_playlist_id,missing_track_id
0,0


**CUSTOMERS, WHERE STATE IS MISSING, SHOW STRING 'NO STATE PROVIDED'**

In [10]:
%%sql
SELECT first_name, last_name, COALESCE(state, 'No State Provided') AS missing_states
FROM customer
WHERE state IS NULL
LIMIT 10;

first_name,last_name,missing_states
Leonie,Köhler,No State Provided
Bjørn,Hansen,No State Provided
František,Wichterlová,No State Provided
Helena,Holý,No State Provided
Astrid,Gruber,No State Provided
Daan,Peeters,No State Provided
Kara,Nielsen,No State Provided
João,Fernandes,No State Provided
Madalena,Sampaio,No State Provided
Hannah,Schneider,No State Provided


**CHECKING FOR DUPLICATES**


CUSTOMER

In [49]:
%%sql
SELECT phone,
       email,
       COUNT(*) AS total_duplicates
FROM customer
GROUP BY phone, email
HAVING COUNT(*) > 1;

phone,email,total_duplicates


ALBUM

In [47]:
%%sql
SELECT title,
       COUNT(*) AS total_duplicates
FROM album
GROUP BY title
HAVING COUNT(*) > 1;

title,total_duplicates


ARTIST

In [45]:
%%sql
SELECT name,
       COUNT(*) AS total_duplicates
FROM artist
GROUP BY name
HAVING COUNT(*) > 1;

name,total_duplicates


EMPLOYEE

In [43]:
%%sql
SELECT email,
     COUNT(*) AS total_duplicates
FROM employee
GROUP BY email
HAVING COUNT(*) > 1;

email,total_duplicates


GENRE

In [36]:
%%sql
SELECT name,
       COUNT(*) AS total_duplicates
FROM genre
GROUP BY name
HAVING COUNT(*) > 1;

name,total_duplicates


INVOICE

In [51]:
%%sql
SELECT customer_id,
       invoice_date,
       COUNT(*) AS total_duplicates
FROM invoice
GROUP BY customer_id, invoice_date
HAVING COUNT(*) > 1;

customer_id,invoice_date,total_duplicates


INVOICE_LINE

In [53]:
%%sql
SELECT invoice_id,
       track_id,
       COUNT(*) AS total_duplicates
FROM invoice_line
GROUP BY invoice_id, track_id
HAVING COUNT(*) > 1;

invoice_id,track_id,total_duplicates


MEDIA TYPE

In [55]:
%%sql
SELECT name,
       COUNT(*) AS total_duplicates
FROM media_type
GROUP BY name
HAVING COUNT(*) > 1;

name,total_duplicates


  PLAYLIST

In [57]:
%%sql
SELECT name,
       COUNT(*) AS total_duplicates
FROM playlist
GROUP BY name
HAVING COUNT(*) > 1;

name,total_duplicates
TV Shows,2
Movies,2
Audiobooks,2
Music,2


> 4 REPEATED PLAYLIST NAMES WERE FOUND  

*ACTUAL PLAYLIST ROWS AND THEIR IDS BEHIND THE REPEATED NAMES*

In [66]:
%%sql
SELECT playlist_id,
       name
FROM playlist
WHERE name IN ('TV Shows', 'Movies', 'Audiobooks', 'Music')
ORDER BY name, playlist_id;

playlist_id,name
4,Audiobooks
6,Audiobooks
2,Movies
7,Movies
1,Music
8,Music
3,TV Shows
10,TV Shows


*HOW MANY SONGS ARE IN EACH ONE*

In [70]:
%%sql
SELECT playlist_id,
       COUNT(*) AS song_count
FROM playlist_track
WHERE playlist_id IN(4,6,2,7,1,8,3,10)
GROUP BY playlist_id
ORDER BY playlist_id;

playlist_id,song_count
1,3290
3,213
8,3290
10,213


Movies and audiobooks are empty, hence they might not be duplicates?



> But same count isn't proof of same songs, so we compare the actual song lists

**CONFIRMING THEY ARE ACTUAL DUPLICATES**

In [63]:
%%sql
#Music (id 1) vs Music (id 8): tracks in one but not the other
  SELECT track_id 
  FROM playlist_track 
  WHERE playlist_id = 1
EXCEPT
  SELECT track_id 
  FROM playlist_track 
  WHERE playlist_id = 8;

#TV Shows (id 3) vs TV Shows (id 10): tracks in one but not the other
  SELECT track_id 
  FROM playlist_track 
  WHERE playlist_id = 3
EXCEPT
  SELECT track_id 
  FROM playlist_track 
  WHERE playlist_id = 10;

track_id


> EXCEPT between playlist id 1 and 8 AND playlist id 3 and 10 found zero different songs between them proving they are not just the same count but the same actual songs

PLAYLIST TRACK

In [60]:
%%sql
SELECT playlist_id
      track_id,
      COUNT(*) AS total_duplicates
FROM playlist_track
GROUP BY track_id, playlist_id
HAVING COUNT(*) > 1;

track_id,total_duplicates


TRACK

In [62]:
%%sql
SELECT name,
       album_id,
       COUNT(*) AS total_duplicates
FROM track
GROUP BY name, album_id
HAVING COUNT(*) > 1;

name,album_id,total_duplicates
Gimme Some Truth,255,2
Imagine,255,2
Banditismo Por Uma Questa,25,2
Branch Closing,251,2
Company Man,228,2
Not In Portland,229,2


**DATA SANITY CHECK**

In [95]:
%%sql
SELECT MIN(total) AS min_total,
       MAX(total) AS max_total,
       ROUND(AVG(total),2) AS avg_total
FROM invoice;

min_total,max_total,avg_total
0.99,25.86,5.65


In [97]:
%%sql
SELECT MIN(quantity) AS min_qty,
       MAX(quantity) AS max_qty,
       MIN(unit_price) AS min_price,
       MAX(unit_price) AS max_price,
       ROUND(AVG(quantity), 2) AS avg_qty
FROM invoice_line;

min_qty,max_qty,min_price,max_price,avg_qty
1,1,0.99,1.99,1.00
